# Midterm Manure Q4 Benchmark

Local VS Code notebook for evaluating ChatbotLP on Midterm Problem 1, Question 4: Manure Management with compost technology.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = ""
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3-flash-preview"

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

from src.midterm_benchmark import (
    DEFAULT_Q4_BENCHMARK_DIR,
    MIDTERM_Q4_REASONING_PROMPTS,
    MidtermBenchmarkConfig,
    load_benchmark_files,
    run_midterm_manure_q4_benchmark,
    write_midterm_outputs,
)

## Problem Statement And Reference Solution

In [ ]:
files = load_benchmark_files(DEFAULT_Q4_BENCHMARK_DIR)
reference = files["reference_solution"]

display(Markdown(files["problem_statement"]))

reference_summary = pd.DataFrame([
    {"metric": "objective_value", "value": reference["objective_value"]},
    {"metric": "demand_revenue", "value": reference["demand_revenue"]},
    {"metric": "supply_contribution", "value": reference["supply_contribution"]},
    {"metric": "transport_cost", "value": reference["transport_cost"]},
    {"metric": "technology_cost", "value": reference["technology_cost"]},
    {"metric": "total_manure_removed", "value": reference["expected_semantic_metrics"]["solver_aggregates"]["total_manure_removed"]},
    {"metric": "total_compost_produced", "value": reference["expected_semantic_metrics"]["solver_aggregates"]["total_compost_produced"]},
])
display(reference_summary)
display(reference)

## Canonical And Paraphrased Prompt Runs

In [ ]:
USE_LLM = bool(os.environ.get("GEMINI_API_KEY"))

config = MidtermBenchmarkConfig(
    prompt_ids=("canonical", "paraphrased"),
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    fallback_to_reference_fixture=True,
    attempt_solve=True,
    run_reasoning=False,
)
report = run_midterm_manure_q4_benchmark(config=config)

display(pd.DataFrame([report["metadata"]]))
display(report["tables"]["case_summary"])
display(report["tables"]["solve_accuracy"])
display(report["tables"]["interpretation_metadata"])
display(report["tables"]["interpretation_errors"])

## ID-Independent Diagnostic Tables

In [ ]:
diagnostic_table_names = [
    "semantic_count_metrics",
    "parameter_multiset_metrics",
    "topology_metrics",
    "technology_yield_metrics",
    "route_economics_metrics",
    "route_association_metrics",
    "solver_aggregate_metrics",
    "balance_residual_metrics",
    "formulation_completeness_metrics",
    "solve_correctness_metrics",
    "reasoning_readiness_metrics",
    "alias_resolution_diagnostics",
]

for table_name in diagnostic_table_names:
    display(Markdown(f"### {table_name.replace('_', ' ').title()}"))
    display(report["tables"][table_name])

## Primary Metric Flags

In [ ]:
primary_flags = report["tables"]["case_summary"][[
    "prompt_id",
    "formulation_completeness_pass",
    "solve_correctness_pass",
    "reasoning_ready_pass",
    "route_association_pass",
    "semantic_structure_pass",
    "technology_structure_pass",
    "solver_aggregate_pass",
    "balance_residual_pass",
    "primary_success",
    "failure_type",
]]
display(primary_flags)

## Q4 Flow And Activity Table

In [ ]:
flow_activity_table = pd.DataFrame([
    {"pathway": "manure to Menomonie/CF", "quantity_tons": 0},
    {"pathway": "manure to Black River Falls/SF", "quantity_tons": 500},
    {"pathway": "manure to Composter", "quantity_tons": 500},
    {"pathway": "compost to Madison/DC", "quantity_tons": 50},
    {"pathway": "technology activity", "quantity_tons": 500},
])
display(flow_activity_table)

## Q3 Comparison

In [ ]:
q3_comparison = pd.DataFrame([
    {"metric": "Q3 objective", "value": 1050},
    {"metric": "Q4 objective", "value": 5800},
    {"metric": "improvement", "value": 4750},
])
display(q3_comparison)

## Required Reasoning Prompts

In [ ]:
display(pd.DataFrame(MIDTERM_Q4_REASONING_PROMPTS)[["id", "label", "prompt"]])

REASONING_PROMPT_ID = None
reasoning_prompt_ids = (REASONING_PROMPT_ID,) if REASONING_PROMPT_ID else None

reasoning_config = MidtermBenchmarkConfig(
    prompt_ids=("canonical",),
    reasoning_prompt_ids=reasoning_prompt_ids,
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    fallback_to_reference_fixture=True,
    attempt_solve=True,
    run_reasoning=True,
)
reasoning_report = run_midterm_manure_q4_benchmark(config=reasoning_config)
display(reasoning_report["tables"]["reasoning_prompt_success"])

for row in reasoning_report["cases"][0]["reasoning_results"]:
    display(Markdown(f"### {row['prompt_label']}

{row['response_preview']}"))

## Export

In [ ]:
output_dir = REPO_ROOT / "midterm_outputs"
write_midterm_outputs(report, output_dir)
reasoning_report["tables"]["reasoning_prompt_success"].to_csv(
    output_dir / "manure_q4_reasoning_prompt_success_canonical.csv",
    index=False,
)
flow_activity_table.to_csv(output_dir / "manure_q4_flow_activity_table.csv", index=False)
q3_comparison.to_csv(output_dir / "manure_q4_q3_comparison.csv", index=False)

paper_summary = report["tables"]["case_summary"][[
    "prompt_id",
    "solver_ready_actual",
    "formulation_completeness_pass",
    "solve_correctness_pass",
    "reasoning_ready_pass",
    "route_association_pass",
    "semantic_structure_pass",
    "technology_structure_pass",
    "solver_aggregate_pass",
    "balance_residual_pass",
    "primary_success",
    "failure_type",
    "solve_success",
]]
paper_summary.to_csv(output_dir / "manure_q4_paper_style_summary.csv", index=False)
display(paper_summary)
output_dir